# Stigmergic Swarm — Colab Runner

Runs the **cleaned pipeline** (`Attempt At Cleaning/`) — the version with strict no-leak
partitioning, the Cluster Genome, composite fitness, and the continuous worker pool.

## Hardware tiers

| Colab tier | GPU | Model auto-selected | Recommended flag |
|---|---|---|---|
| Free | T4 16 GB | Qwen2.5-7B-Instruct (fp16) | `--small` for 3B if OOM |
| Pro | L4 24 GB | Qwen2.5-14B-Instruct (fp16) | — |
| Pro+ | A100 40 GB | Qwen2.5-32B (small bundle) | — |
| Pro+ | A100 80 GB | Qwen2.5-32B-Instruct (bf16) | `--bundle=debate_analysis` for multi-engine |
| Pro+ | H100 80 GB | Qwen2.5-32B-Instruct (bf16) | `--bundle=debate_analysis` |

**`--small`** forces the `small` bundle: `Qwen2.5-3B-Instruct` + `Phi-3.5-mini`. Fits on any tier including T4. Use it if you hit OOM or want a faster iteration loop.

## Before you start

> **Push your local changes to GitHub first.**
> Cell 3 clones `https://github.com/sfuqua6/Stigmeric-Coordination.git`.
> If your local branch is ahead of `origin/main`, the notebook will run **stale code**.
> Run `git push` locally before opening this notebook.

## Execution order

Run cells top-to-bottom the first time. On a reconnected session, re-run cells 1–4
(mount, paths, clone/pull, install) — everything else is idempotent.

Cell 7 (MOCK_LLM sanity check) runs without a GPU and is always safe to re-run.

## Cell 1 — Mount Google Drive

Run outputs, knowledge base, and retrieval cache land on Drive so they survive
Colab session resets. The HuggingFace model cache stays in `/content/hf_cache`
(ephemeral — re-downloaded each new session, but HF Hub caches by hash so
re-download is fast on a warm CDN).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — Configure paths

All persistent data writes under `DRIVE_BASE`. Change `swarm` to any subfolder name you prefer.

In [ ]:
import os

DRIVE_BASE = '/content/drive/MyDrive/swarm'

os.environ['SWARM_OUTPUTS_BASE_DIR']    = f'{DRIVE_BASE}/runs'
os.environ['SWARM_KB_DIR']              = f'{DRIVE_BASE}/knowledge_base'
os.environ['SWARM_RETRIEVAL_CACHE_DIR'] = f'{DRIVE_BASE}/retrieval_cache'
os.environ['SWARM_CORPORA_DIR']         = f'{DRIVE_BASE}/corpora'

for d in ('runs', 'knowledge_base', 'retrieval_cache', 'corpora'):
    os.makedirs(f'{DRIVE_BASE}/{d}', exist_ok=True)

# Ephemeral HF model cache (refilled from HF CDN each session).
os.environ['HF_HOME']            = '/content/hf_cache'
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.makedirs('/content/hf_cache', exist_ok=True)

# Tell config.py we're on Colab so tier-detection fires.
os.environ['COLAB'] = '1'

# Silence library chatter (transformers load bars, BertModel reports, etc.).
os.environ['SWARM_QUIET_LIBS'] = '1'

print('Persistent dirs (Drive):')
for k in ('SWARM_OUTPUTS_BASE_DIR', 'SWARM_KB_DIR',
          'SWARM_RETRIEVAL_CACHE_DIR', 'SWARM_CORPORA_DIR'):
    print(f'  {os.environ[k]}')
print(f'HF_HOME: {os.environ["HF_HOME"]}')

## Cell 3 — Clone / pull the repository

First run: clones to `/content/swarm_repo`.
Subsequent runs on the same session: `git pull` to pick up any new pushes.

> **Reminder:** push your local commits before running this cell.

In [ ]:
import os, subprocess

REPO_URL  = 'https://github.com/sfuqua6/Stigmeric-Coordination.git'
REPO_ROOT = '/content/swarm_repo'
# The cleaned pipeline lives in the 'Attempt At Cleaning' subdirectory.
REPO_WORKDIR = f'{REPO_ROOT}/Attempt At Cleaning'

if not os.path.exists(REPO_ROOT):
    subprocess.run(['git', 'clone', REPO_URL, REPO_ROOT], check=True)
else:
    print(f'{REPO_ROOT} exists — running git pull')
    subprocess.run(['git', '-C', REPO_ROOT, 'pull'], check=True)

out = subprocess.run(
    ['git', '-C', REPO_ROOT, 'log', '--oneline', '-5'],
    capture_output=True, text=True
)
print('Recent commits:')
print(out.stdout)

# All subsequent shell commands use REPO_WORKDIR.
os.chdir(REPO_WORKDIR)
print('cwd:', os.getcwd())

## Cell 4 — Install dependencies

~5 minutes on a cold Colab instance (vllm downloads a CUDA-matched torch wheel).
Subsequent sessions are faster because pip's wheel cache is warm.

`requirements-colab.txt` installs: `vllm`, `sentence-transformers`, `bitsandbytes`,
`cohere`, `datasets`, `faiss-cpu`, `wikipedia`, `ddgs`, `requests`, `beautifulsoup4`, `tqdm`.

In [ ]:
!pip install -q -r requirements-colab.txt

# Verify the important ones loaded.
import importlib
for pkg in ('vllm', 'sentence_transformers', 'bitsandbytes', 'faiss'):
    try:
        m = importlib.import_module(pkg)
        ver = getattr(m, '__version__', 'ok')
        print(f'  {pkg}: {ver}')
    except ImportError as e:
        print(f'  {pkg}: MISSING — {e}')

## Cell 5 — GPU info and tier detection

Confirms which GPU Colab assigned and which model `config.py` will auto-select.
The tier drives model choice, dtype, and worker concurrency without any manual flags.

In [ ]:
!nvidia-smi --query-gpu=name,memory.free,memory.total --format=csv

import sys
sys.path.insert(0, '.')

from core import config
print()
print(f'Detected tier : {config._TIER!r}')
print(f'Model selected: {config.MODEL_NAME!r}')
print(f'VLLM dtype    : {config.VLLM_DTYPE!r}')
print(f'LLM_CONCURRENCY: {config.LLM_CONCURRENCY}')
print()
print('If the tier or model is wrong, override with:')
print('  import os; os.environ["SWARM_TIER"] = "t4"  # t4 | l4 | a100_40 | a100_80 | h100')
print('  import os; os.environ["SWARM_MODEL"] = "Qwen/Qwen2.5-7B-Instruct"')

## Cell 6 — API keys (optional, but enables better retrieval)

Without keys the pipeline falls back to Wikipedia HTTP + DuckDuckGo, which works
fine. With a Cohere key the primary retrieval path (pre-computed Wikipedia embeddings
via `CohereCorpusRetriever`) fires and quality improves.

- **Cohere key** (free): https://cohere.com/
- **HF token** (free read-only): https://huggingface.co/settings/tokens

In [ ]:
import os

# === FILL IN OR LEAVE BLANK ===
COHERE_API_KEY = ''   # paste key here, or leave blank to use Wikipedia/DDG fallback
HF_TOKEN       = ''   # read-only HF token for downloading model weights from gated repos
# ==============================

if COHERE_API_KEY:
    os.environ['COHERE_API_KEY'] = COHERE_API_KEY
    print('COHERE_API_KEY: set')
else:
    print('COHERE_API_KEY: not set — will fall back to Wikipedia/DDG retrieval')

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
    print('HF_TOKEN: set')
else:
    print('HF_TOKEN: not set — fine for public models (Qwen, Phi)')

## Cell 7 — Sanity check (no GPU / no model download required)

`MOCK_LLM=1` skips model loading entirely. MockLLM emits deterministic SHA1-seeded
phrases — proves the pipeline wiring (signal store, worker pool, synthesizer, outputs)
without downloading anything. Always safe to run.

Expected: a `outputs_mock/RUN_.../answer.txt` file appears and no Python exceptions are raised.

In [ ]:
!MOCK_LLM=1 SWARM_MIN_TIME_S=0 SWARM_MIN_ITERATIONS=5 SWARM_MAX_ITERATIONS=20 \
    python run_swarm.py debate "Cities should ban private cars" --corpus=placeholder

## Cell 8 — Real runs

### Which cell to run?

| Situation | Use |
|---|---|
| **T4 free tier** or want a fast run | Cell 8a (`--small`) |
| **L4 / A100 / H100** and want best quality | Cell 8b (auto-tier) |
| **A100 80 GB / H100** and want multi-engine routing | Cell 8c (`--bundle`) |

First run downloads model weights to `/content/hf_cache/` (~5–30 min depending on model size).
Subsequent runs load from cache (<2 min).

> The pipeline runs as a **shell subprocess** (`!python run_swarm.py ...`), not in-kernel.
> This is required for vLLM's subprocess engine core to initialise properly.
> Do **not** call `run_continuous_pipeline()` directly from the kernel — it will hang.

In [ ]:
# Cell 8a — T4 / small bundle (Qwen2.5-3B + Phi-3.5-mini, fits on 16 GB)
# Change the task type and prompt as needed.
# Task types: debate | analysis | creative | problem_solving | coding

TASK   = 'debate'
PROMPT = 'Cities should ban private cars to fight climate change.'

!python run_swarm.py "{TASK}" "{PROMPT}" --small --workers=8

In [ ]:
# Cell 8b — Auto-tier selection (L4 → 14B, A100/H100 → 32B)
# Config.py reads the GPU name and selects model + dtype automatically.

TASK   = 'debate'
PROMPT = 'Cities should ban private cars to fight climate change.'

!python run_swarm.py "{TASK}" "{PROMPT}" --workers=24

In [ ]:
# Cell 8c — Multi-engine bundle (A100 80 GB or H100 required)
# debate_analysis bundle: 14B primary + 32B-AWQ reasoner + 7B fast
# All three engines stay resident; roles are routed per-engine.
#
# Available bundles:
#   debate_analysis  — debate / analysis tasks  (needs ~80 GB)
#   coding           — coding tasks             (needs ~80 GB)
#   creative         — creative tasks           (needs ~40 GB)
#   small            — 3B + Phi-3.5, any tier   (needs ~6 GB)
#   small_coding     — 3B coder + Phi-3.5       (needs ~6 GB)

TASK   = 'debate'
PROMPT = 'Cities should ban private cars to fight climate change.'
BUNDLE = 'debate_analysis'

!python run_swarm.py "{TASK}" "{PROMPT}" --bundle={BUNDLE} --workers=24

In [ ]:
# Cell 8d — Coding task
# On T4: use --small (routes to small_coding bundle: Qwen2.5-Coder-3B)
# On A100: omit --small (routes to coding or a100_coding bundle automatically)

PROMPT = 'Implement a Python function that returns the longest palindromic substring in O(n^2) time.'

!python run_swarm.py coding "{PROMPT}" --small --workers=8

## Cell 9 — Run the test suite\n\n322 tests pass, 10 skip (as of the last commit). Runs entirely with `MOCK_LLM=1`\n— no GPU or model download needed. Takes ~6 min.\n\nConvergence env vars disable the 60-second wall-clock gate so subprocess tests\ndon't time out inside pytest.

In [ ]:
!MOCK_LLM=1 SWARM_MIN_TIME_S=0 SWARM_MIN_ITERATIONS=5 SWARM_MAX_ITERATIONS=20 \
    pytest tests/ -q --tb=short 2>&1 | tail -20

## Cell 10 — Inspect the most recent run

Every run writes to a timestamped subdirectory:

| File | Contents |
|---|---|
| `answer.txt` | Final synthesized answer |
| `summary.json` | Run metadata + diversity metrics |
| `signals.json` | Full surviving signal DAG with genome fields |
| `round_log.json` | Per-round strength dynamics, output diversity |
| `citations.json` | Sourced claims with signal IDs |
| `lineage.dot` | DOT graph of signal ancestry (render with Graphviz) |
| `renderer_audit.json` | Faithfulness audit (4-gram overlap per citation) |
| `run_meta.json` | Config snapshot (model, tier, flags, genome stats) |

In [ ]:
import json, os
from pathlib import Path

outputs_base = os.environ.get('SWARM_OUTPUTS_BASE_DIR', 'outputs')
outputs_root = Path(outputs_base)
if not outputs_root.exists():
    outputs_root = Path('outputs')

runs = sorted(outputs_root.glob('*'), key=lambda p: p.stat().st_mtime) if outputs_root.exists() else []

if not runs:
    print(f'No runs found in {outputs_root}.')
    print('Run a real-model cell (8a / 8b / 8c / 8d) first.')
else:
    latest = runs[-1]
    print(f'Latest run: {latest}\n')

    answer_path = latest / 'answer.txt'
    if answer_path.exists():
        print('=== answer.txt ===')
        print(answer_path.read_text())
        print()

    summary_path = latest / 'summary.json'
    if summary_path.exists():
        summary = json.loads(summary_path.read_text())
        print('=== summary.json (selected fields) ===')
        interesting = ['task_type', 'model', 'tier', 'n_surviving_clusters',
                       'composite_fitness_mean', 'support_diversity_mean',
                       'genome_atoms_total', 'grounding_mean']
        for k in interesting:
            if k in summary:
                print(f'  {k}: {summary[k]}')
        print()

## Cell 11 — Compare two runs

Side-by-side diff of `summary.json` fields between two output directories.

In [ ]:
import os
from pathlib import Path

outputs_base = os.environ.get('SWARM_OUTPUTS_BASE_DIR', 'outputs')
outputs_root = Path(outputs_base)
if not outputs_root.exists():
    outputs_root = Path('outputs')

runs = sorted(outputs_root.glob('*'), key=lambda p: p.stat().st_mtime) if outputs_root.exists() else []

if len(runs) < 2:
    print('Need at least 2 runs to compare. Run cell 8 twice with different settings.')
else:
    run_a, run_b = str(runs[-2]), str(runs[-1])
    print(f'Comparing:\n  A: {run_a}\n  B: {run_b}\n')
    !python tools/compare_runs.py "{run_a}" "{run_b}"

## Cell 12 — Useful one-liners

Copy-paste as needed.

In [ ]:
# Run pipeline diagnostics (signal-store self-check, no LLM needed).
# !MOCK_LLM=1 python diagnose.py

# Re-render synthesis from a saved signals.json (no re-running the pipeline).
# !python synthesize.py outputs/RUN_TIMESTAMP

# Migrate knowledge base from schema v2 to v3 (adds genome fields to old entries).
# !python kb_migrate.py

# Run with the placeholder corpus (skip live retrieval — faster, reproducible).
# !python run_swarm.py debate "Your thesis here" --corpus=placeholder --small

# Run in baseline mode (no signal store, independent agents — A/B comparison).
# !python run_swarm.py debate "Your thesis here" --mode=baseline --small

# Reset knowledge base before run (quarantine existing KB entries).
# !python run_swarm.py debate "Your thesis here" --reset-kb --small

# Override tier manually if auto-detection picks the wrong GPU.
# import os; os.environ['SWARM_TIER'] = 'l4'

# Force a specific model (bypasses tier-based auto-selection).
# import os; os.environ['SWARM_MODEL'] = 'Qwen/Qwen2.5-14B-Instruct'

print('Uncomment the line you want and run this cell.')

## Troubleshooting

### OOM during model load
Add `--small` to the run command (cell 8a / 8d). The small bundle uses ~6 GB VRAM and fits any tier.

### `Engine core initialization failed. Failed core proc(s): {}`
You tried to run the pipeline in-kernel (e.g., `await run_continuous_pipeline(...)`).
vLLM's subprocess engine-core cannot handshake inside the Jupyter kernel.
Always use `!python run_swarm.py ...` (the shell prefix).

### `ImportError: cannot import name 'make_bundle_router'`
Your cloned repo is stale. Re-run cell 3 to `git pull`.

### Tier detected as `unknown` or wrong model selected
Force the tier manually before running:
```python
import os; os.environ['SWARM_TIER'] = 't4'  # or l4 / a100_40 / a100_80 / h100
```
Then re-run cell 5 to confirm and proceed to cell 8.

### Tests fail with timeout errors
The convergence env vars (`SWARM_MIN_TIME_S=0 SWARM_MIN_ITERATIONS=5`) must be set.
Cell 9 already sets them. If running pytest manually, prepend them.

### `AssertionError: partition_id` in a deposit call
A new agent role was added without supplying `partition_id` in deposit metadata.
See `CLAUDE.md` → Partition invariant section for the fix.

### Notebook is out of date with repo
Re-run cell 3 (`git pull`). The notebook itself does not auto-update — it is a static
launcher. The pipeline code under `/content/swarm_repo/` is what you pulled.

### `SpeculativeConfig validation error`
Your `SWARM_SPECULATIVE_DRAFT` env var points to a model with a mismatched tokenizer.
Unset it: `import os; del os.environ['SWARM_SPECULATIVE_DRAFT']`. Speculative decoding
is off by default.